In [0]:
-- ============================================================================
-- TASK 2: VALIDATE STAGING
-- ============================================================================
-- PURPOSE  : Runs 4 automated checks on the staging table before it is
--            promoted to live. If any check fails, the query raises an error
--            which causes the Databricks Workflow task to fail and blocks
--            Task 3 (the swap) from running entirely.
--
-- CHECKS   :
--   1. Empty check     — staging must have at least 1 row
--   2. Row count check — staging must have >= 90% of rows vs live table
--                        (guards against upstream CDC lag / source outages)
--   3. Schema check    — all columns in live must exist in staging
--                        (guards against upstream schema drift)
--   4. Null check      — key business columns must not be entirely NULL
--                        (guards against silent join failures)
--
-- DEPENDS ON : task1_write_staging.sql (set as dependency in Workflow)
-- NEXT TASK  : task3_swap_and_cleanup.sql
-- ============================================================================


-- ----------------------------------------------------------------------------
-- CHECK 1: Staging must not be empty
-- ----------------------------------------------------------------------------
SELECT
  CASE
    WHEN COUNT(*) = 0
    THEN RAISE_ERROR('VALIDATION FAILED: Staging table is empty. Swap blocked.')
  END
FROM furlenco_analytics.materialized_tables.fin_metric_query_growth_staging;


-- ----------------------------------------------------------------------------
-- CHECK 2: Staging row count must be >= 90% of live table row count
-- Adjust 0.90 threshold if your data has known seasonal dips > 10%
-- ----------------------------------------------------------------------------
SELECT
  CASE
    WHEN staging_count < (live_count * 0.90)
    THEN RAISE_ERROR(
      CONCAT(
        'VALIDATION FAILED: Row count check failed. ',
        'Staging rows: ', CAST(staging_count AS STRING),
        ' | Live rows: ',  CAST(live_count    AS STRING),
        ' | Threshold (90%): ', CAST(CAST(live_count * 0.90 AS BIGINT) AS STRING),
        '. Swap blocked.'
      )
    )
  END
FROM (
  SELECT
    (SELECT COUNT(*) FROM furlenco_analytics.materialized_tables.fin_metric_query_growth_staging) AS staging_count,
    (SELECT COUNT(*) FROM furlenco_analytics.materialized_tables.fin_metric_query_growth)         AS live_count
);


-- ----------------------------------------------------------------------------
-- CHECK 3: Schema match — no columns should be missing from staging vs live
-- Catches upstream column drops or renames that would break downstream queries
-- ----------------------------------------------------------------------------
SELECT
  CASE
    WHEN COUNT(*) > 0
    THEN RAISE_ERROR(
      CONCAT(
        'VALIDATION FAILED: Schema mismatch detected. ',
        'Columns in live but missing from staging: ',
        ARRAY_JOIN(COLLECT_LIST(column_name), ', '),
        '. Swap blocked.'
      )
    )
  END
FROM (
  SELECT column_name
  FROM information_schema.columns
  WHERE table_catalog = 'furlenco_analytics'
    AND table_schema  = 'materialized_tables'
    AND table_name    = 'fin_metric_query_growth'
  EXCEPT
  SELECT column_name
  FROM information_schema.columns
  WHERE table_catalog = 'furlenco_analytics'
    AND table_schema  = 'materialized_tables'
    AND table_name    = 'fin_metric_query_growth_staging'
);


-- ----------------------------------------------------------------------------
-- CHECK 4: Key business columns must not be entirely NULL
-- Guards against silent upstream join failures producing all-NULL outputs
-- Add or remove columns here based on what your team considers critical
-- ----------------------------------------------------------------------------
SELECT
  CASE
    WHEN order_id_nulls  = total_rows THEN RAISE_ERROR('VALIDATION FAILED: order_id is entirely NULL in staging.')
    WHEN entity_id_nulls = total_rows THEN RAISE_ERROR('VALIDATION FAILED: entity_id is entirely NULL in staging.')
    WHEN fur_id_nulls    = total_rows THEN RAISE_ERROR('VALIDATION FAILED: fur_id is entirely NULL in staging.')
    WHEN ebvmr_nulls     = total_rows THEN RAISE_ERROR('VALIDATION FAILED: ebvmr is entirely NULL in staging.')
  END
FROM (
  SELECT
    COUNT(*)                          AS total_rows,
    COUNT_IF(order_id   IS NULL)      AS order_id_nulls,
    COUNT_IF(entity_id  IS NULL)      AS entity_id_nulls,
    COUNT_IF(fur_id     IS NULL)      AS fur_id_nulls,
    COUNT_IF(ebvmr      IS NULL)      AS ebvmr_nulls
  FROM furlenco_analytics.materialized_tables.fin_metric_query_growth_staging
);


-- ----------------------------------------------------------------------------
-- All checks passed — log staging stats for the Workflow run audit trail
-- ----------------------------------------------------------------------------
SELECT
  'VALIDATION PASSED'                                                           AS status,
  (SELECT COUNT(*) FROM furlenco_analytics.materialized_tables.fin_metric_query_growth_staging) AS staging_row_count,
  (SELECT COUNT(*) FROM furlenco_analytics.materialized_tables.fin_metric_query_growth)         AS live_row_count,
  current_timestamp()                                                           AS validated_at;